In [1]:
!pip install -q google-generativeai
!pip install -q chromadb
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 74.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [5]:
import os
import google.generativeai as genai
import chromadb
import numpy as np

from kaggle_secrets import UserSecretsClient

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

In [6]:
user_secrets=UserSecretsClient()

API_KEY=user_secrets.get_secret(
"GEMINI_API_KEY"
)

genai.configure(
api_key=API_KEY
)

print("Gemini Connected")

Gemini Connected


In [9]:
os.makedirs(
"/kaggle/working/company_docs",
exist_ok=True
)

In [10]:
docs={

"vacation_policy.txt":
"""
Employees receive
20 paid vacation days.
Unused leave expires.
""",

"remote_work.txt":
"""
Employees can work
from home 3 days weekly.
""",

"maternity_leave.txt":
"""
Employees receive
90 days maternity leave.
"""
}

for file,text in docs.items():

    with open(
        f"/kaggle/working/company_docs/{file}",
        "w"
    ) as f:

        f.write(text)

print("Files Created")

Files Created


In [12]:
!pip install -U google-generativeai

In [14]:
from google import genai

client = genai.Client(
    api_key=API_KEY
)


def get_embedding(text):

    response = client.models.embed_content(

        model="gemini-embedding-001",

        contents=text

    )

    return response.embeddings[0].values


text = "vacation policy"

embedding = get_embedding(text)

print(
    "Length:",
    len(embedding)
)

print(
    embedding[:5]
)

Length: 3072
[-0.019459445, 0.02876737, -0.0052722213, -0.048846874, -0.021738686]


In [19]:
import google.generativeai as genai


def get_embedding(text):

    response = genai.embed_content(

        model="models/gemini-embedding-001",

        content=text

    )

    return response["embedding"]


text="vacation policy"

embedding=get_embedding(text)

print(
"Length:",
len(embedding)
)

print(
embedding[:5]
)

Length: 3072
[-0.019459445, 0.02876737, -0.0052722213, -0.048846874, -0.021738686]


In [15]:
def cosine(v1,v2):

    v1=np.array(v1)

    v2=np.array(v2)

    return (
        np.dot(v1,v2)
        /
        (
            np.linalg.norm(v1)
            *
            np.linalg.norm(v2)
        )
    )

phrases=[

"vacation policy",

"time off rules",

"PTO guidelines",

"dress code"

]

emb=[]

for p in phrases:

    emb.append(
        get_embedding(p)
    )

base=emb[0]

for i in range(
1,
len(phrases)
):

    score=cosine(
        base,
        emb[i]
    )

    print(
phrases[i],
score
)

time off rules 0.7293365072548121
PTO guidelines 0.6689141994595184
dress code 0.5763880310730993


In [16]:
client=chromadb.PersistentClient(

path=
"/kaggle/working/chroma_db"

)

collection=client.get_or_create_collection(

name=
"company_docs"

)

print(
collection.count()
)

0


In [17]:
loader=DirectoryLoader(

"/kaggle/working/company_docs",

glob="*.txt",

loader_cls=TextLoader

)

documents=loader.load()

splitter=RecursiveCharacterTextSplitter(

chunk_size=500,

chunk_overlap=50

)

chunks=splitter.split_documents(
documents
)

print(
len(chunks)
)

3


In [20]:
for i, chunk in enumerate(chunks):

    emb = get_embedding(
        chunk.page_content
    )

    collection.add(

        ids=[str(i)],

        documents=[
            chunk.page_content
        ],

        embeddings=[
            emb
        ]

    )

print(
"Indexed Successfully"
)

Indexed Successfully


In [21]:
def vector_search(query):

    emb=get_embedding(
query
)

    return collection.query(

query_embeddings=[emb],

n_results=2

    )

result=vector_search(

"time off policy"

)

print(
result["documents"]
)

[['Employees receive\n20 paid vacation days.\nUnused leave expires.', 'Employees can work\nfrom home 3 days weekly.']]


In [22]:
model=genai.GenerativeModel(
"gemini-2.5-flash"
)

def rag(query):

    result=vector_search(
query
)

    context="\n".join(
result["documents"][0]
)

    prompt=f"""

Answer using ONLY context.

Context:
{context}

Question:
{query}

"""

    response=model.generate_content(
prompt
)

    return response.text

In [23]:
questions=[

"How much vacation?",

"Can I work from home?",

"What is maternity leave?"

]

for q in questions:

    print("\nQ:",q)

    print(
rag(q)
)


Q: How much vacation?
20 paid vacation days.

Q: Can I work from home?
Yes, employees can work from home 3 days weekly.

Q: What is maternity leave?
Maternity leave is 90 days.
